### Title: Get federal lobbying records

output: CSV data

author: Suhail Bhat

Method: Use US congress's REST API to fetch records dynamically. REST API is a way for programs to exchange data over http, where each kind of thing lives at its own URL

Documentation: https://lda.gov/api/v1/

Variables: https://lda.gov/api/redoc/v1/#tag/Filings/operation/listFilings

In [ ]:
import requests

In [ ]:
url = "https://lda.gov//api/v1/filings/"

headers={"Accept": "application/json"}

response = requests.get(url, headers=headers)

print(response.status_code)

In [ ]:
data=response.json()

print(data.keys())

In [ ]:
filing = data['results'][0]

print(filing.keys())

In [ ]:
print(filing["client"]["name"])
print(filing["registrant"]["name"])
print(filing["income"], filing["expenses"])

In [ ]:
for activity in filing["lobbying_activities"]:
    print(activity["general_issue_code_display"])
    print(activity["description"])

#### Anthropic and OpenAI records

In [ ]:
url="https://lda.gov/api/v1/clients/"

r = requests.get(
    "https://lda.gov/api/v1/clients/",
    params={"client_name": "anthropic"},
    headers={"Accept": "application/json"},
)

In [ ]:
for c in r.json()["results"]:
    print(c['registrant']['name'])

#### Filings by lobbyists

In [ ]:
import time

search_names=["anthropic", "openai"]
headers= {"Accept": "application/json"}

all_filings= []


## Search each name. results come 25 at a time, so we need to loop through pages of results until we get them all
for name in search_names:
    print(f"Fetching filings for {name}")
    url = "https://lda.gov/api/v1/filings/"
    params = {"client_name": name, "page_size": 25}

    while url:
        response = requests.get(url, params=params, headers = headers)

        data = response.json()

        all_filings = all_filings + data['results']
        print("  Downloaded", len(data["results"]), "of", data["count"], "filings")

        ## Link to the next page of results, if it exists
        url= data['next']
        params = None  # Don't send params on subsequent pages, as the next URL already has them
        time.sleep(1)

In [ ]:
## turn each filings into one simple row

rows = []

for filing in all_filings:

    ## skips registrations; they announce a new client but report no money
    if "Registration" in filing["filing_type_display"]:
        continue

    # Outside firms report "income"; AI firms's own staff report "expenses"
    amount = filing["income"]
    if amount is None:
        amount = filing["expenses"]

    ## collect the issues listed on this report
    issues = []
    details = []
    officials = []
    for activity in filing["lobbying_activities"]:
        issues.append(activity["general_issue_code_display"])
        details.append(activity["description"] or "")
        # officials.append(activity["lobbyists"])
        for lobbyist in activity["lobbyists"]:
            position = lobbyist["covered_position"]
            if position is None or position == "N/A":
                continue
            officials.append(lobbyist['covered_position'])

    rows.append({
        "filing_id": filing["filing_uuid"],
        "year": filing["filing_year"],
        "quarter": filing["filing_period_display"],
        "report_type": filing["filing_type_display"],
        "client": filing["client"]["name"],
        "lobbying_firm": filing["registrant"]["name"],
        "amount": amount,
        "issues": " | ".join(issues),
        "details": " | ".join(details),
        "officials": " | ".join(officials),
        "link": filing["filing_document_url"],
    })


In [ ]:
## save in a dataframe
import pandas as pd
df=pd.DataFrame(rows)

print("Rows before duplication", df.shape)

df=df.drop_duplicates(subset=["filing_id"], keep="first")

print("Rows after duplication", df.shape)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df["amount"]=df['amount'].astype(float)

In [ ]:
df.groupby("year")['amount'].sum().sort_values(ascending=False)

In [ ]:
df.groupby(["year","quarter"])['amount'].sum().reset_index().sort_values('year', ascending=False)

In [ ]:
df["q"] = df["quarter"].str[0].astype(int)

In [ ]:
# 2. Sort oldest to newest: by year first, then by quarter
df = df.sort_values(["year", "q"])

In [ ]:
df["label"] = df["year"].astype(str) + " Q" + df["q"].astype(str)


In [ ]:
import matplotlib.pyplot as plt
plt.style.use("fivethirtyeight")

In [ ]:
df.groupby("year")['amount'].sum().reset_index()

In [ ]:
df.groupby(["label"])['amount'].sum().reset_index()

In [ ]:
df.groupby(["label"])['amount'].sum().reset_index().plot(
    x="label", y="amount", kind="barh", 
    title="AI federal lobbying spending by quarter", 
    xlabel="Amount ($)", ylabel="Quarter", figsize=(10, 6), legend=False)

In [ ]:
df.to_csv("../output/ai-lobbying.csv", index=False)

## Grab by issue

In [ ]:
## "Gaming/Gambling/Casino"

In [ ]:
base_url="https://lda.gov"
api_endpoint="/api/v1/constants/filing/lobbyingactivityissues/"

url=base_url+api_endpoint

url

In [ ]:
headers={"Accept": "application/json"}

response=requests.get(url, headers=headers)

In [ ]:
import pandas as pd
issues_data= pd.DataFrame(response.json())

issues_data.shape

In [ ]:
filings_url ="https://lda.gov/api/v1/filings/"

filings_url

In [ ]:
headers={"Accept": "application/json"}

response=requests.get(filings_url, headers=headers)

data = response.json()

In [ ]:
data.keys(), len(data['results'])

In [ ]:
### Three things:

### client_name, industry, 

import time

url = "https://lda.gov/api/v1/filings/"
params = {"filing_specific_lobbying_issues": "super intelligence","page_size": 25}

all_filings = []

while url:
    response = requests.get(url, params=params, headers = headers)

    data = response.json()

    all_filings = all_filings + data['results']
    print("  Downloaded", len(data["results"]), "of", data["count"], "filings")

    ## Link to the next page of results, if it exists
    url= data['next']
    params = None  # Don't send params on subsequent pages, as the next URL already has them
    time.sleep(1)

In [ ]:
rows=[]

for filing in all_filings:
    rows.append({
        "period":filing['filing_period_display'],
        "year": filing['filing_year'],
        "client_name": filing['client']['name'],
        "lobbying_goals": filing["lobbying_activities"][0]['general_issue_code_display'] ,
        "amount": filing['income']
        })

In [ ]:
pd.DataFrame(rows)